In [394]:
import pandas as pd
import numpy as np

In [395]:
df = pd.read_csv("../data/raw/Dados Históricos_Wesley.csv", parse_dates=[0], index_col=0, decimal=',', thousands='.') # Alteração: ajustando a colunda de data e colocando como index 
df.index = pd.to_datetime(df.index, format = "%d.%m.%Y")
df = df.sort_index()

In [396]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 500 entries, 2024-03-01 to 2026-03-02
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Último    500 non-null    int64
 1   Abertura  500 non-null    int64
 2   Máxima    500 non-null    int64
 3   Mínima    500 non-null    int64
 4   Vol.      500 non-null    str  
 5   Var%      500 non-null    str  
dtypes: int64(4), str(2)
memory usage: 27.3 KB


In [397]:
df.dropna()

,Último,Abertura,Máxima,Mínima,Vol.,Var%
Data,,,,,,
2024-03-01,129180,129026,129716,128717,"9,77M","0,12%"
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%"
2024-03-05,128098,128336,128989,127823,"9,69M","-0,19%"
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%"
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%"
...,...,...,...,...,...,...
2026-02-24,191490,188854,191781,188854,"11,04B","1,40%"
2026-02-25,191247,191491,192624,190419,"9,13B","-0,13%"
2026-02-26,191005,191248,191978,188977,"8,80B","-0,13%"


## Definição do Target

O target é definido com base na variação diária do preço de fechamento.

In [398]:
# Retorno percentual do dia
retorno = df["Último"].pct_change()

# Threshold: só considera alta/baixa se a variação for maior que 0.3%
# Dias com variação pequena são descartados (ruído)
threshold = 0.003

df["y"] = np.where(retorno > threshold, 1,
          np.where(retorno < -threshold, 0, np.nan))

df = df.dropna()

print("Distribuição do novo target:")
print(df["y"].value_counts(normalize=True).round(3))
print("Dias descartados (ruído):", retorno.notna().sum() - df["y"].notna().sum())

Distribuição do novo target:
y
1.0    0.546
0.0    0.454
Name: proportion, dtype: float64
Dias descartados (ruído): 151


Este procedimento transforma a variação diária em uma variável binária:

- 1 → dia de alta

- 0 → dia de queda

A primeira linha da série é removida devido à ausência de um valor anterior para cálculo da diferença.

In [399]:
df[["Último","y"]].head(10)

,Último,y
Data,,
2024-03-04,128341,0.0
2024-03-06,128890,1.0
2024-03-07,128340,0.0
2024-03-08,127071,0.0
2024-03-11,126124,0.0
2024-03-12,127668,1.0
2024-03-15,126742,0.0
2024-03-19,127529,1.0
2024-03-20,129125,1.0


In [400]:
df["y"].value_counts(normalize=True)

y
1.0    0.545977
0.0    0.454023
Name: proportion, dtype: float64

## Engenharia de Atributos (Feature Engineering)
A engenharia de atributos tem como objetivo transformar os dados brutos de preço em variáveis que capturem padrões relevantes do comportamento do mercado.

Todas as features são defasadas em um período (shift(1)) para evitar data leakage, garantindo que apenas informações disponíveis até o dia anterior sejam utilizadas na previsão.

### Retornos Recentes

In [401]:
df["retorno_1"] = df["Último"].pct_change().shift(1)
df["retorno_3"] = df["Último"].pct_change(3).shift(1)

**Interpretação**

Retornos representam a variação percentual do preço.

Essas variáveis capturam o comportamento recente do mercado.

**retorno_1**

Representa o retorno percentual do último pregão.

In [402]:
df["vol_5"] = df["Último"].pct_change().rolling(5).std().shift(1)
df["vol_10"] = df["Último"].pct_change().rolling(10).std().shift(1)

In [403]:
df

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10
Data,,,,,,,,,,,
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%",0.0,NaN,NaN,NaN,NaN
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%",1.0,NaN,NaN,NaN,NaN
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%",0.0,0.004278,NaN,NaN,NaN
2024-03-08,127071,128335,128338,125802,"11,94M","-0,99%",0.0,-0.004267,NaN,NaN,NaN
2024-03-11,126124,127068,127068,126065,"8,90M","-0,75%",0.0,-0.009888,-0.009896,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-19,188534,186020,188687,185928,"9,23B","1,35%",1.0,-0.006934,0.001197,0.013500,0.014092
2026-02-20,190534,188531,190727,186700,"9,99B","1,06%",1.0,0.011101,-0.006141,0.013757,0.013986
2026-02-23,188853,190532,191003,188526,"9,90B","-0,88%",0.0,0.010608,0.014742,0.012502,0.013404


### Médias Móveis

In [404]:
# Calculando a diferença da tendencias
df["ma_5"] = df["Último"].rolling(5).mean().shift(1)

df["ma_10"] = df["Último"].rolling(10).mean().shift(1)
df["ma_diff"] = df["ma_5"] - df["ma_10"]

**Interpretação**

Médias móveis para identificar tendências.

- MA5 → tendência de curtíssimo prazo

- MA10 → tendência de curto prazo

In [405]:
df

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff
Data,,,,,,,,,,,,,,
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%",1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%",0.0,0.004278,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-08,127071,128335,128338,125802,"11,94M","-0,99%",0.0,-0.004267,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-11,126124,127068,127068,126065,"8,90M","-0,75%",0.0,-0.009888,-0.009896,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-19,188534,186020,188687,185928,"9,23B","1,35%",1.0,-0.006934,0.001197,0.013500,0.014092,186624.0,184779.3,1844.7
2026-02-20,190534,188531,190727,186700,"9,99B","1,06%",1.0,0.011101,-0.006141,0.013757,0.013986,187740.8,185319.3,2421.5
2026-02-23,188853,190532,191003,188526,"9,90B","-0,88%",0.0,0.010608,0.014742,0.012502,0.013404,188599.4,186236.3,2363.1


### Diferença entre Médias Móveis

In [406]:
# calculando a diferença de range
df["range_diff"] = (df["Máxima"] - df["Mínima"]).shift(1)

In [407]:
# 1. Distâncias Relativas (Substituindo ma_5 e ma_10 brutos)
df["dist_ma_5"] = ((df["Último"] - df["Último"].rolling(5).mean()) / df["Último"].rolling(5).mean()).shift(1)
df["dist_ma_10"] = ((df["Último"] - df["Último"].rolling(10).mean()) / df["Último"].rolling(10).mean()).shift(1)

# 2. Pressão de Compra/Venda (Onde fechou dentro do candle anterior)
df["pressao_fechamento"] = ((df["Último"] - df["Mínima"]) / (df["Máxima"] - df["Mínima"] + 1e-9)).shift(1)

Essa variável mede a distância entre duas médias móveis.

**Interpretação**

| Valor | Significado |
|------|-------------|
| Positivo | Tendência de alta |
| Negativo | Tendência de baixa |



In [408]:
df["momentum_5"] = (df["Último"] - df["Último"].shift(5)).shift(1)

In [409]:
df.dropna(inplace=True)

As primeiras linhas da série é removida devido à ausência de um valor anterior para cálculo da diferença.

## Separando teste e treino

Diferentemente de problemas tradicionais de machine learning, séries temporais não devem utilizar divisão aleatória.

O conjunto de teste deve conter os dados mais recentes.

Conforme especificado no desafio, os últimos 30 dias foram reservados para teste.

In [410]:
# criando base de teste com as ultimas 30 linhas
test = df.tail(30)
test

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff,range_diff,dist_ma_5,dist_ma_10,pressao_fechamento,momentum_5
Data,,,,,,,,,,,,,,,,,,,
2025-12-30,161125,160491,162075,160491,"5,87B","0,40%",1.0,0.012513,0.019889,0.014082,0.017987,158551.4,159063.6,-512.2,2312.0,0.012013,0.008754,1.000000,-2026.0
2026-01-02,160539,161124,161957,160059,"7,46B","-0,36%",0.0,0.004169,0.020276,0.007264,0.010889,159060.8,159439.2,-378.4,1584.0,0.012977,0.010573,0.400253,2547.0
2026-01-05,161870,160542,162166,160215,"7,99B","0,83%",1.0,-0.003637,0.013037,0.005725,0.010996,159703.2,159674.4,28.8,1898.0,0.005233,0.005415,0.252898,3212.0
2026-01-06,163664,161870,164135,161870,"8,41B","1,11%",1.0,0.008291,0.008812,0.006018,0.011138,160492.6,159953.9,538.7,1951.0,0.008582,0.011979,0.848283,3947.0
2026-01-07,161975,163661,163661,161746,"8,85B","-1,03%",0.0,0.011083,0.015758,0.006491,0.011179,161530.8,160243.7,1287.1,2265.0,0.013206,0.021344,0.792053,5191.0
2026-01-08,162936,161975,162936,161748,"8,61B","0,59%",1.0,-0.010320,0.008945,0.008808,0.011307,161834.6,160193.0,1641.6,1915.0,0.000868,0.011124,0.119582,1519.0
2026-01-13,161973,163146,163146,161765,"9,67B","-0,72%",0.0,0.005933,0.006586,0.008955,0.007703,162196.8,160628.8,1568.0,1188.0,0.004557,0.014364,1.000000,1811.0
2026-01-14,165146,161974,165146,161974,"9,93B","1,96%",1.0,-0.005910,-0.010332,0.009377,0.007420,162483.6,161093.4,1390.2,1381.0,-0.003142,0.005460,0.150615,1434.0
2026-01-16,164800,165557,165872,164100,"11,49B","-0,46%",0.0,0.019590,0.019577,0.012249,0.009111,163138.8,161815.7,1323.1,3172.0,0.012304,0.020581,1.000000,3276.0


In [411]:
# montando base de treino sem as ultimas 30 linhas
train = df.iloc[:-30]
train

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff,range_diff,dist_ma_5,dist_ma_10,pressao_fechamento,momentum_5
Data,,,,,,,,,,,,,,,,,,,
2024-03-27,127691,126863,127756,126223,"9,92M","0,65%",1.0,-0.008833,-0.003936,0.009709,0.008892,127716.4,127667.5,48.9,1280.0,-0.005398,-0.005017,0.115625,-641.0
2024-03-28,128106,127689,128364,127270,"9,94M","0,33%",1.0,0.005227,-0.011106,0.009285,0.008959,127906.2,127547.6,358.6,1533.0,-0.001682,0.001124,0.957599,949.0
2024-04-01,126990,128106,128659,126772,"9,37M","-0,87%",0.0,0.003250,-0.000414,0.009002,0.008961,128021.6,127524.2,497.4,1094.0,0.000659,0.004562,0.764168,577.0
2024-04-02,127549,126990,127654,126669,"9,07M","0,44%",1.0,-0.008712,-0.000291,0.006946,0.008825,127594.6,127516.1,78.5,1887.0,-0.004738,-0.004126,0.115527,-2135.0
2024-04-05,126795,127422,127432,126394,"9,10M","-0,50%",0.0,0.004402,-0.001112,0.007191,0.008508,127472.6,127658.6,-186.0,985.0,0.000599,-0.000859,0.893401,-610.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-16,158578,162482,162482,158558,"9,92B","-2,40%",0.0,0.010674,0.027151,0.023013,0.017369,159575.8,160280.9,-705.1,2307.0,0.018212,0.013733,0.743823,-1974.0
2025-12-17,157327,158578,158611,156351,"11,34B","-0,79%",0.0,-0.024027,-0.003124,0.014575,0.018818,159817.6,160283.2,-465.6,3924.0,-0.007756,-0.010639,0.005097,1209.0
2025-12-18,157923,157327,158495,157124,"7,80B","0,38%",1.0,-0.007889,-0.021391,0.014941,0.018945,159645.6,160108.7,-463.1,2260.0,-0.014523,-0.017374,0.431858,-860.0


## Definição das Variáveis de Entrada

In [412]:
# momentum normalizado pelo nível de preço (comparável entre épocas)
df["momentum_5_norm"] = df["momentum_5"] / df["Último"]

# range normalizado (volatilidade relativa do candle)
df["range_norm"] = df["range_diff"] / df["Último"].shift(1)

features = ["retorno_1", "retorno_3", "ma_10", "ma_5", "ma_diff", "momentum_5", "range_diff", "dist_ma_5", "dist_ma_10", "pressao_fechamento"]

# definindo x e y de treino
X_train = train[features]
y_train = train["y"]

# definindo x e y de teste
X_test = test[features]
y_test = test["y"]

## Modelos e Metricas Avaliadas 

In [413]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [414]:
from sklearn.linear_model import LogisticRegression

model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train)

pred_log = model_log.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_log))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_log))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_log))

Accuracy: 0.5333333333333333

Matriz de confusão:  [[ 0 11]
 [ 3 16]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        11
         1.0       0.59      0.84      0.70        19

    accuracy                           0.53        30
   macro avg       0.30      0.42      0.35        30
weighted avg       0.38      0.53      0.44        30



In [415]:
from sklearn.tree import DecisionTreeClassifier

model_tree = DecisionTreeClassifier(random_state=42)
model_tree.fit(X_train, y_train)

pred_tree = model_tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_tree))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_tree))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_tree))

Accuracy: 0.5

Matriz de confusão:  [[ 8  3]
 [12  7]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.40      0.73      0.52        11
         1.0       0.70      0.37      0.48        19

    accuracy                           0.50        30
   macro avg       0.55      0.55      0.50        30
weighted avg       0.59      0.50      0.49        30



In [416]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf))

Accuracy: 0.6

Matriz de confusão:  [[ 6  5]
 [ 7 12]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.46      0.55      0.50        11
         1.0       0.71      0.63      0.67        19

    accuracy                           0.60        30
   macro avg       0.58      0.59      0.58        30
weighted avg       0.62      0.60      0.61        30



In [417]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [418]:
from sklearn.svm import SVC

model_svm = SVC()
model_svm.fit(X_train_scaled, y_train)

pred_svm = model_svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_svm))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_svm))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_svm))

Accuracy: 0.6

Matriz de confusão:  [[ 0 11]
 [ 1 18]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        11
         1.0       0.62      0.95      0.75        19

    accuracy                           0.60        30
   macro avg       0.31      0.47      0.38        30
weighted avg       0.39      0.60      0.47        30



In [419]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [420]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

pred_knn = model_knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_knn))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_knn))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_knn))

Accuracy: 0.5666666666666667

Matriz de confusão:  [[ 1 10]
 [ 3 16]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.25      0.09      0.13        11
         1.0       0.62      0.84      0.71        19

    accuracy                           0.57        30
   macro avg       0.43      0.47      0.42        30
weighted avg       0.48      0.57      0.50        30



In [421]:
from sklearn.ensemble import GradientBoostingClassifier

model_gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model_gb.fit(X_train, y_train)

pred_gb = model_gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_gb))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_gb))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_gb))

Accuracy: 0.43333333333333335

Matriz de confusão:  [[ 5  6]
 [11  8]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.31      0.45      0.37        11
         1.0       0.57      0.42      0.48        19

    accuracy                           0.43        30
   macro avg       0.44      0.44      0.43        30
weighted avg       0.48      0.43      0.44        30



In [422]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model_xgb.fit(X_train, y_train)

pred_xgb = model_xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_xgb))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_xgb))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_xgb))

Accuracy: 0.7333333333333333

Matriz de confusão:  [[ 6  5]
 [ 3 16]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.67      0.55      0.60        11
         1.0       0.76      0.84      0.80        19

    accuracy                           0.73        30
   macro avg       0.71      0.69      0.70        30
weighted avg       0.73      0.73      0.73        30



/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:09:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [423]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [424]:
resultados = {
    "Logistic Regression": accuracy_score(y_test, pred_log),
    "Decision Tree": accuracy_score(y_test, pred_tree),
    "Random Forest": accuracy_score(y_test, pred_rf),
    "KNN": accuracy_score(y_test, pred_knn),
    "SVM": accuracy_score(y_test, pred_svm),
    "Gradient Boosting": accuracy_score(y_test, pred_gb),
    "XGBoost": accuracy_score(y_test, pred_xgb)
}

pd.Series(resultados).sort_values(ascending=False)

XGBoost                0.733333
Random Forest          0.600000
SVM                    0.600000
KNN                    0.566667
Logistic Regression    0.533333
Decision Tree          0.500000
Gradient Boosting      0.433333
dtype: float64

In [425]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [426]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

print("Melhores parâmetros:", grid_rf.best_params_)
print("Melhor acurácia no CV:", round(grid_rf.best_score_, 4))

pred_rf_tuned = grid_rf.best_estimator_.predict(X_test)

print("\nAcurácia no teste:", accuracy_score(y_test, pred_rf_tuned))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf_tuned))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf_tuned))

Melhores parâmetros: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 300}
Melhor acurácia no CV: 0.5137

Acurácia no teste: 0.6333333333333333

Matriz de confusão:  [[ 0 11]
 [ 0 19]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        11
         1.0       0.63      1.00      0.78        19

    accuracy                           0.63        30
   macro avg       0.32      0.50      0.39        30
weighted avg       0.40      0.63      0.49        30



/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0

In [427]:
print("Distribuição do target no TREINO:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribuição do target no TESTE:")
print(y_test.value_counts(normalize=True).round(3))

print("\nDatas do conjunto de teste:")
print(test.index[0], "até", test.index[-1])
print("Total de dias:", len(test))

Distribuição do target no TREINO:
y
1.0    0.544
0.0    0.456
Name: proportion, dtype: float64

Distribuição do target no TESTE:
y
1.0    0.633
0.0    0.367
Name: proportion, dtype: float64

Datas do conjunto de teste:
2025-12-30 00:00:00 até 2026-02-27 00:00:00
Total de dias: 30


In [428]:
correlacoes = X_train.copy()
correlacoes["y"] = y_train

print("Correlação de cada feature com o target (treino):")
print(correlacoes.corr()["y"].drop("y").sort_values(key=abs, ascending=False).round(4))

Correlação de cada feature com o target (treino):
ma_5                  0.0865
ma_10                 0.0797
momentum_5            0.0743
ma_diff               0.0735
range_diff           -0.0698
dist_ma_10            0.0620
pressao_fechamento    0.0393
retorno_3             0.0355
dist_ma_5             0.0336
retorno_1            -0.0089
Name: y, dtype: float64


In [429]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0]
}

grid_xgb = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    param_grid=param_grid_xgb,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1
)
grid_xgb.fit(X_train, y_train)

print("Melhores parâmetros:", grid_xgb.best_params_)
print("Melhor acurácia no CV:", round(grid_xgb.best_score_, 4))

pred_xgb_tuned = grid_xgb.best_estimator_.predict(X_test)

print("\nAcurácia no teste:", accuracy_score(y_test, pred_xgb_tuned))
print('\nMatriz de confusão:\n', confusion_matrix(y_test, pred_xgb_tuned))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_xgb_tuned))

/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:09:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:09:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:09:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/pre

Melhores parâmetros: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100, 'subsample': 1.0}
Melhor acurácia no CV: 0.5216

Acurácia no teste: 0.6333333333333333

Matriz de confusão:
 [[ 1 10]
 [ 1 18]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.50      0.09      0.15        11
         1.0       0.64      0.95      0.77        19

    accuracy                           0.63        30
   macro avg       0.57      0.52      0.46        30
weighted avg       0.59      0.63      0.54        30



/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [17:09:25] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
